# Python `str.strip()`: Master Boundary Whitespace & Character Trimming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_strip_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.strip() Tutorial](https://www.plus2net.com/python/string-strip.php)**.

### Learning Objectives:
1. Master default whitespace trimming across ASCII and Unicode boundaries.
2. Understand the character-set stripping mechanic when passing custom `chars`.
3. Avoid the classic suffix destruction trap by comparing `rstrip()` with `removesuffix()` (PEP 616).
4. Clean tabular CSV fields, normalize NLP tokens, and collapse internal whitespace.
5. Validate implementations with automated self-checking unit assertions.

## 1. Default Whitespace Trimming

When called with no arguments or `None`, `strip()` removes all leading and trailing spaces, tabs (`\t`), and newlines (`\n`, `\r`).
Internal spaces between words remain untouched.

In [ ]:
raw_input = "  \t\r\n  user_name_123  \n\t  "

# 1. Clean boundary whitespace
clean = raw_input.strip()
print("Clean:   ", repr(clean))
assert clean == "user_name_123"

# 2. Internal spaces are strictly preserved
phrase = "  hello    world  "
assert phrase.strip() == "hello    world"

# 3. Empty string
assert "".strip() == ""

print("\nAll default whitespace assertions passed!")


## 2. The Custom Character Set Mechanic

The `chars` argument is treated as an unordered **set of characters**, NOT a substring.
Python strips any character in the set until it reaches a character not in the set.

In [ ]:
banner = "...---=== TITLE ===---..."

# Characters in set: '.', '-', '=', and ' '
clean_title = banner.strip(".-= ")
print("Clean title:", repr(clean_title))
assert clean_title == "TITLE"

# Order of characters in set does not matter
assert banner.strip("=-. ") == "TITLE"

print("\nCharacter set assertions passed!")


## 3. The Classic Trap: `rstrip()` vs. `removesuffix()` (PEP 616)

Using `rstrip(".txt")` on `"test.txt"` destroys the trailing `'t'` of `'test'` because `'t'` is in `{'.', 't', 'x'}`.
Always use `removesuffix()` for exact substring removals.

In [ ]:
filename = "test.txt"

# Flawed: rstrip() treats '.txt' as a character set
broken = filename.rstrip(".txt")
print("rstrip('.txt') result:      ", repr(broken))  # Produces 'tes'!
assert broken == "tes"

# Correct: removesuffix() removes the exact literal suffix
correct = filename.removesuffix(".txt")
print("removesuffix('.txt') result:", repr(correct)) # Produces 'test'
assert correct == "test"

print("\nSuffix trap contrast passed!")


## 4. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.strip()`:
1. Tabular Cell Sanitization (Quotes, Currencies, Whitespace)
2. NLP Token Punctuation Normalization
3. Internal Whitespace Collapsing Pipeline

In [ ]:
import string

# Recipe 1: Tabular cell cleaner
def clean_cell(val: str) -> str:
    return val.strip(" \t\r\n'\"$€£")

assert clean_cell('  "$1,450.00"  ') == "1,450.00"
assert clean_cell(" 'Active' \n") == "Active"

# Recipe 2: NLP token cleaning
def clean_token(word: str) -> str:
    return word.strip(string.punctuation).lower()

raw_tokens = ['"Hello,', 'state-of-the-art!', "(Python's)", '...end.']
cleaned = [clean_token(t) for t in raw_tokens]
print("Recipe 2 Tokens:", cleaned)
assert cleaned == ["hello", "state-of-the-art", "python's", "end"]

# Recipe 3: Whitespace collapsing
def collapse_whitespace(text: str) -> str:
    return " ".join(text.split())

query = "   SELECT   id,    name   FROM   users   "
collapsed = collapse_whitespace(query)
print("Recipe 3 SQL:   ", collapsed)
assert collapsed == "SELECT id, name FROM users"

print("\nAll engineering recipes passed!")


## 5. Production Implementation & Self-Checking Unit Assertions

This production-grade `TextTrimmer` class encapsulates boundary whitespace stripping, character set removal, and safe suffix trimming.

In [ ]:
class TextTrimmer:
    @staticmethod
    def clean_boundary_whitespace(text: str) -> str:
        """Removes all leading and trailing ASCII/Unicode whitespace."""
        return text.strip()

    @staticmethod
    def strip_chars(text: str, char_set: str) -> str:
        """Strips designated characters from both ends."""
        return text.strip(char_set)

    @staticmethod
    def safe_remove_extension(filename: str, ext: str) -> str:
        """Demonstrates correct suffix removal vs flawed rstrip."""
        return filename.removesuffix(ext)


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
trimmer = TextTrimmer()

# 1. Boundary whitespace
assert trimmer.clean_boundary_whitespace("   hello world   \n\t") == "hello world"
assert trimmer.clean_boundary_whitespace("") == ""

# 2. Custom character set
assert trimmer.strip_chars("### Welcome to Python ###", "# ") == "Welcome to Python"
assert trimmer.strip_chars("...---Data---...", ".-") == "Data"

# 3. Safe suffix removal vs rstrip
target_file = "test.txt"
assert target_file.rstrip(".txt") == "tes"
assert trimmer.safe_remove_extension(target_file, ".txt") == "test"

# 4. TypeError on non-string
try:
    "test".strip(123)
    assert False, "Expected TypeError not raised"
except TypeError as e:
    assert "strip arg must be None or str" in str(e)

print("All str.strip() production unit assertions passed successfully!")
